# Education Improvement Predictive

## Problem Framing

**Business question:** Which residents with active education records are most likely to improve soon?

**Operational decision supported:** Help staff reinforce the education plans and support patterns linked to improvement.

**Primary users:** case managers, education coordinators

**Target summary:** Current predictive label: `label_education_improvement_next_120d`, based on future attendance gains, progress gains, or a shift to completed status.

This standardized predictive notebook uses the shared notebook factory so every pipeline follows the same submission structure and evaluation flow.


## Shared Assets And Notebook Bootstrap

Shared references:

* `ml/docs/data-joins.md`
* `ml/docs/feature-catalog.md`
* `ml/docs/phase-3-predictive-pipelines.md`
* `ml/docs/phase-4-modeling-framework.md`
* `ml/docs/phase-b-notebook-standardization.md`


In [1]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "ml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


In [2]:
from ml.src.pipelines.notebook_support import (
    load_evaluation_bundle,
    load_notebook_context,
    summarize_frame,
)

context = load_notebook_context(
    pipeline_name='education_improvement',
    dataset_name='resident_monthly_features',
    predictive_impl='education_improvement',
    use_predictive_dataset=True,
)
pipeline_name = context["pipeline_name"]
dataset_name = context["dataset_name"]
config = context["config"]
dataset = context["dataset"]

summarize_frame(dataset)


,resident_id,snapshot_month,safehouse_id,case_status,case_category,sex,initial_risk_level,months_since_admission,age_years_at_snapshot,subcategory_flag_count,...,future_window_complete_60d,future_window_complete_90d,future_window_complete_120d,label_incident_next_30d,label_case_prioritization_next_60d,label_counseling_progress_next_90d,label_education_improvement_next_120d,label_wellbeing_deterioration_next_90d,label_supportive_home_visit_next_120d,label_reintegration_complete_next_90d
0,50,2023-01-01,4,Active,Neglected,F,High,0,11.90,2,...,True,True,True,False,False,False,1,False,False,False
1,13,2023-02-01,2,Closed,Surrendered,F,Medium,0,8.73,1,...,True,True,True,False,True,True,1,False,False,False
2,45,2023-02-01,3,Transferred,Abandoned,F,Medium,0,14.99,2,...,True,True,True,False,False,False,1,False,True,False
3,50,2023-02-01,4,Active,Neglected,F,High,1,11.98,2,...,True,True,True,False,False,False,1,False,False,False
4,23,2023-02-01,5,Closed,Foundling,F,High,0,9.07,1,...,True,True,True,False,True,True,1,False,False,False
5,2,2023-03-01,3,Closed,Surrendered,F,Medium,0,14.85,2,...,True,True,True,False,False,False,1,False,True,False
6,13,2023-03-01,2,Closed,Surrendered,F,Medium,1,8.80,1,...,True,True,True,False,True,True,1,False,False,False
7,36,2023-03-01,1,Active,Surrendered,F,High,0,12.06,2,...,True,True,True,False,True,False,1,False,False,False
8,29,2023-03-01,8,Transferred,Abandoned,F,Medium,0,12.08,1,...,True,True,True,False,True,True,1,False,False,False
9,45,2023-03-01,3,Transferred,Abandoned,F,Medium,1,15.07,2,...,True,True,True,False,False,False,1,False,True,False


## Target And Leakage Checklist

1. Restate the target in business terms: Current predictive label: `label_education_improvement_next_120d`, based on future attendance gains, progress gains, or a shift to completed status.
2. Confirm the snapshot date or split column before running any new model fits.
3. Remove fields that directly encode the future target or post-outcome information.
4. Record any threshold, calibration, or class-balance choice that changes deployment behavior.


## Standard Model Comparison Outputs

Every predictive notebook should read the same evaluation bundle before writing conclusions:

* saved metrics JSON
* Phase 4 holdout comparison table
* Phase 4 cross-validation summary


In [3]:
evaluation = load_evaluation_bundle('education_improvement')
metrics = evaluation["metrics"]
holdout_comparison = evaluation["holdout_comparison"]
cv_summary = evaluation["cv_summary"]

metrics


{'best_model_name': 'logistic_regression',
 'train_rows': 756,
 'test_rows': 653,
 'target_col': 'label_education_improvement_next_120d',
 'split_col': 'snapshot_month',
 'selection_metric': 'average_precision',
 'cutoff_date': '2025-04-01',
 'task_type': 'classification',
 'sample_count': 653.0,
 'positive_count': 47.0,
 'positive_rate': 0.07197549770290965,
 'accuracy': 0.9571209800918836,
 'balanced_accuracy': 0.8591391053999017,
 'precision': 0.6862745098039216,
 'recall': 0.7446808510638298,
 'f1': 0.7142857142857143,
 'roc_auc': 0.9841303279264098,
 'average_precision': 0.8045457485928328}

In [4]:
summarize_frame(holdout_comparison)


,pipeline_name,model_name,sample_count,positive_count,positive_rate,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision,mae,rmse,r2
0,education_improvement,logistic_regression,653.0,47.0,0.071975,0.957121,0.859139,0.686275,0.744681,0.714286,0.984130,0.804546,NaN,NaN,NaN
1,education_improvement,random_forest_classifier,653.0,47.0,0.071975,0.961715,0.822361,0.775000,0.659574,0.712644,0.983481,0.775911,NaN,NaN,NaN
2,education_improvement,dummy_classifier,653.0,47.0,0.071975,0.928025,0.500000,0.000000,0.000000,0.000000,0.500000,0.071975,NaN,NaN,NaN


In [5]:
summarize_frame(cv_summary)


,pipeline_name,model_name,fold_mean,fold_std,sample_count_mean,sample_count_std,positive_count_mean,positive_count_std,positive_rate_mean,positive_rate_std,...,roc_auc_std,average_precision_mean,average_precision_std,n_splits,mae_mean,mae_std,rmse_mean,rmse_std,r2_mean,r2_std
0,education_improvement,random_forest_classifier,3.0,1.581139,151.2,0.447214,63.0,0.0,0.41667,0.001228,...,0.015909,0.932619,0.025284,5,NaN,NaN,NaN,NaN,NaN,NaN
1,education_improvement,logistic_regression,3.0,1.581139,151.2,0.447214,63.0,0.0,0.41667,0.001228,...,0.010449,0.897691,0.019031,5,NaN,NaN,NaN,NaN,NaN,NaN
2,education_improvement,dummy_classifier,3.0,1.581139,151.2,0.447214,63.0,0.0,0.41667,0.001228,...,0.000000,0.416670,0.001228,5,NaN,NaN,NaN,NaN,NaN,NaN


## Business Interpretation

Write the final narrative in plain language:

1. What does a high score mean operationally for this pipeline?
2. Which staff action should happen next because of the score?
3. Which users should trust the ranking signal versus wait for more threshold work?
4. Which fairness, bias, or data-quality caveats need to be called out to case managers, education coordinators?


## Deployment Notes

Recommended shared widgets:

* `insight_summary_card`
* `ranked_table_widget`
* `recommendation_panel`

Deployment checklist:

* Show the score in education planning views and student support summaries.
* Pair improvement scores with a short explanation so staff can distinguish traction from stall risk.

Standard endpoint pattern:

* `POST /ml/predict/education_improvement`
* `POST /ml/score-batch/education_improvement`
